# Source Database Setup
### In this Notebook, we will setup the source database, which is a PostgreSQL database. We will create the database, tables, and insert some sample data. 

In [ ]:
CREATE OR REPLACE DATABASE CONNECTOR_DEST_DB;
GRANT CREATE SCHEMA ON DATABASE CONNECTOR_DEST_DB TO APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL;
ALTER APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL SET SHARE_EVENTS_WITH_PROVIDER = TRUE;

In [ ]:
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_DATA_SOURCE('PSQLDS1', 'CONNECTOR_DEST_DB');
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'raw_cdc', ARRAY_CONSTRUCT('customers'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'raw_cdc', ARRAY_CONSTRUCT('merchants'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'raw_cdc', ARRAY_CONSTRUCT('products'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'raw_cdc', ARRAY_CONSTRUCT('transactions'));

In [ ]:
SELECT * FROM SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.REPLICATION_STATE;

In [ ]:
SELECT * FROM SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.CONNECTOR_STATS;

In [ ]:
SELECT * FROM CONNECTOR_DEST_DB."raw_cdc"."customers";

In [ ]:
SELECT * FROM CONNECTOR_DEST_DB."raw_cdc"."merchants";

In [ ]:
SELECT * FROM CONNECTOR_DEST_DB."raw_cdc"."products";

In [ ]:
SELECT * FROM CONNECTOR_DEST_DB."raw_cdc"."transactions";

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE cdc_prod.analytics.customer_purchase_summary
TARGET_LAG = '1 minute' 
WAREHOUSE = X_SMALL_WH
AS
SELECT
    t.transaction_id
    , t.customer_id
    , c.age AS customer_age
    , t.product_id
    , p.product_name
    , p.product_category
    , t.merchant_id
    , m.merchant_name
    , m.merchant_category
    , t.transaction_date
    , t.transaction_time
    , t.quantity
    , t.quantity * p.price AS total_price
    , t.transaction_card
    , t.transaction_category
FROM
    CONNECTOR_DEST_DB."raw_cdc"."transactions" t
JOIN
    CONNECTOR_DEST_DB."raw_cdc"."customers" c ON t.customer_id = c.customer_id
JOIN
    CONNECTOR_DEST_DB."raw_cdc"."products" p ON t.product_id = p.product_id
JOIN
    CONNECTOR_DEST_DB."raw_cdc"."merchants" m ON t.merchant_id = m.merchant_id
AND
    m.merchant_category = p.product_category;

SELECT * FROM cdc_prod.analytics.customer_purchase_summary;

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP USER IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL_AGENT_USER;

DROP ROLE IF EXISTS CDC_DATA_SCIENTIST;
DROP ROLE IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL_AGENT_ROLE;
DROP ROLE IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL_ADMINISTRATIVE_AGENT_ROLE;
DROP ROLE IF EXISTS POSTGRESQL_ADMINISTRATIVE_AGENT_ROLE;
DROP ROLE IF EXISTS POSTGRESQL_AGENT_ROLE;

DROP APPLICATION IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL CASCADE;

DROP WAREHOUSE IF EXISTS CONNECTOR_DEST_DB;
DROP WAREHOUSE IF EXISTS CDC_DS_WH;
DROP WAREHOUSE IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL_COMPUTE_WH;
DROP WAREHOUSE IF EXISTS SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL_OPS_WH;

DROP DATABASE IF EXISTS CDC_PROD;
DROP DATABASE IF EXISTS CONNECTOR_DEST_DB;